# 🚀 Credit Scoring Optimized — Baseline Model Tuning

## Mục tiêu
Cải thiện AUC của baseline model (hiện tại **0.767**) mà **KHÔNG thêm bảng phụ nào**.

## Chiến lược tối ưu

| # | Optimization | Expected Gain | Effort |
|---|-------------|---------------|--------|
| 1 | **Thêm features từ application_train.csv** (đang chỉ dùng 34/122 cột) | +0.005-0.01 | Thấp |
| 2 | **Thêm feature engineering** (interactions, polynomial, binning) | +0.003-0.008 | Thấp |
| 3 | **Target Encoding** thay LabelEncoder cho high-cardinality cols | +0.002-0.005 | Thấp |
| 4 | **Hyperparameter Tuning** với Optuna (Bayesian optimization) | +0.005-0.01 | TB |
| 5 | **Ensemble** LightGBM + XGBoost + CatBoost | +0.005-0.015 | TB |

**Target: AUC ≥ 0.78** chỉ từ `application_train.csv`.

## 1. Import & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import joblib
import gc
from pathlib import Path

from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

print('Libraries loaded!')

In [ ]:
import os
os.chdir(os.path.dirname(os.path.abspath('__file__')))

DATA_PATH = '../input/application_train.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape}')
print(f'Target: {df["TARGET"].value_counts().to_dict()}')
print(f'Default rate: {df["TARGET"].mean():.4f}')
print(f'\nTotal columns available: {df.shape[1]}')

## 2. Feature Selection — Mở rộng từ 34 → ~55 raw features

So với baseline cũ (34 raw features), thêm:
- **Housing features** (APARTMENTS, LIVINGAREA, FLOORSMAX...) — proxy tài sản
- **Document flags** (FLAG_DOCUMENT_3/6/8) — mức độ cung cấp giấy tờ
- **Credit bureau inquiry count** — số lần hỏi vay gần đây
- **Region mismatch flags** — đăng ký khác nơi ở/làm việc
- **Application timing** — giờ nộp đơn, thứ trong tuần

In [ ]:
# ============================================================
# FEATURE SELECTION: Mở rộng từ baseline
# ============================================================

# === GIỮ NGUYÊN từ baseline (34 features) ===
BASELINE_FEATURES = [
    # Nhân khẩu học
    'CODE_GENDER', 'DAYS_BIRTH', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    # Tài chính
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_CONTRACT_TYPE',
    # Việc làm
    'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH',
    # Tài sản
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE',
    # Khu vực
    'NAME_HOUSING_TYPE', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY',
    'REGION_POPULATION_RELATIVE',
    # External
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    # Social & Contact
    'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL',
    'DAYS_LAST_PHONE_CHANGE',
]

# === MỚI: Thêm features từ application_train (không cần bảng phụ) ===
NEW_RAW_FEATURES = [
    # Thông tin nhà ở (proxy tài sản — tất cả có sẵn trong application_train)
    'APARTMENTS_AVG',                 # Diện tích căn hộ (normalized)
    'LIVINGAREA_AVG',                 # Diện tích ở (normalized)
    'FLOORSMAX_AVG',                  # Số tầng tối đa tòa nhà
    'YEARS_BUILD_AVG',                # Năm xây dựng
    'TOTALAREA_MODE',                 # Tổng diện tích (mode)
    'EMERGENCYSTATE_MODE',            # Tình trạng khẩn cấp tòa nhà
    
    # Document flags (mức độ cung cấp giấy tờ → proxy độ tin cậy)
    'FLAG_DOCUMENT_3',                # Giấy tờ phổ biến nhất
    'FLAG_DOCUMENT_6',
    'FLAG_DOCUMENT_8',
    
    # Credit bureau inquiries (SỐ LẦN HỎI VAY gần đây — rất quan trọng!)
    'AMT_REQ_CREDIT_BUREAU_HOUR',     # Hỏi trong giờ qua
    'AMT_REQ_CREDIT_BUREAU_DAY',      # Hỏi trong ngày qua
    'AMT_REQ_CREDIT_BUREAU_WEEK',     # Hỏi trong tuần qua
    'AMT_REQ_CREDIT_BUREAU_MON',      # Hỏi trong tháng qua
    'AMT_REQ_CREDIT_BUREAU_QRT',      # Hỏi trong quý qua
    'AMT_REQ_CREDIT_BUREAU_YEAR',     # Hỏi trong năm qua
    
    # Region mismatch (đăng ký khác nơi ở/làm việc → rủi ro)
    'REG_REGION_NOT_LIVE_REGION',
    'REG_REGION_NOT_WORK_REGION',
    'REG_CITY_NOT_LIVE_CITY',
    'REG_CITY_NOT_WORK_CITY',
    'LIVE_CITY_NOT_WORK_CITY',
    
    # Application timing
    'HOUR_APPR_PROCESS_START',        # Giờ nộp đơn (nộp đêm khuya = red flag?)
    
    # Social observation
    'OBS_30_CNT_SOCIAL_CIRCLE',       # Số người quan sát 30d
    'OBS_60_CNT_SOCIAL_CIRCLE',       # Số người quan sát 60d
    
    # Contact
    'FLAG_CONT_MOBILE',               # Có SĐT di động liên lạc được
    
    # Who accompanied
    'NAME_TYPE_SUITE',                # Ai đi cùng khi nộp đơn
]

ALL_RAW_FEATURES = BASELINE_FEATURES + NEW_RAW_FEATURES

# Kiểm tra tất cả đều có trong dataset
missing_cols = [c for c in ALL_RAW_FEATURES if c not in df.columns]
if missing_cols:
    print(f'⚠️ Missing columns: {missing_cols}')
else:
    print(f'✅ Tất cả {len(ALL_RAW_FEATURES)} raw features đều có trong dataset')
    print(f'   Baseline: {len(BASELINE_FEATURES)} | Mới: {len(NEW_RAW_FEATURES)}')

## 3. Data Cleaning

In [ ]:
# ============================================================
# DATA CLEANING
# ============================================================
data = df[['SK_ID_CURR', 'TARGET'] + ALL_RAW_FEATURES].copy()

# 1. Loại bỏ giới tính XNA
data = data[data['CODE_GENDER'] != 'XNA']

# 2. DAYS_EMPLOYED: 365243 là placeholder
data['DAYS_EMPLOYED'] = data['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3. DAYS_LAST_PHONE_CHANGE: 0 là placeholder
data['DAYS_LAST_PHONE_CHANGE'] = data['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan)

# 4. Loại outlier thu nhập cực đoan
data = data[data['AMT_INCOME_TOTAL'] < 20_000_000]

print(f'Data shape after cleaning: {data.shape}')
print(f'\nMissing values (top 15):')
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(1)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False).head(15))

## 4. Feature Engineering — Mở rộng

Giữ nguyên 14 features cũ + thêm ~15 features mới:
- **EXT_SOURCE interactions** (quan trọng nhất!)
- **Credit bureau inquiry aggregates**
- **Region mismatch score**
- **Document count**
- **Annuity-to-goods, credit-to-goods ratios**
- **Income binning**

In [ ]:
# ============================================================
# FEATURE ENGINEERING — BASELINE (giữ nguyên)
# ============================================================

# --- Chuyển đổi thời gian ---
data['AGE_YEARS'] = (-data['DAYS_BIRTH'] / 365.25).round(1)
data['EMPLOYMENT_YEARS'] = (-data['DAYS_EMPLOYED'] / 365.25).round(1)
data['REGISTRATION_YEARS'] = (-data['DAYS_REGISTRATION'] / 365.25).round(1)
data['ID_PUBLISH_YEARS'] = (-data['DAYS_ID_PUBLISH'] / 365.25).round(1)
data['PHONE_CHANGE_DAYS'] = -data['DAYS_LAST_PHONE_CHANGE']

# --- Tỷ số tài chính (giữ nguyên) ---
data['CREDIT_INCOME_RATIO'] = data['AMT_CREDIT'] / data['AMT_INCOME_TOTAL']
data['ANNUITY_INCOME_RATIO'] = data['AMT_ANNUITY'] / data['AMT_INCOME_TOTAL']
data['CREDIT_TERM_MONTHS'] = data['AMT_CREDIT'] / data['AMT_ANNUITY']
data['PAYMENT_RATE'] = data['AMT_ANNUITY'] / data['AMT_CREDIT']
data['INCOME_PER_PERSON'] = data['AMT_INCOME_TOTAL'] / data['CNT_FAM_MEMBERS']
data['GOODS_CREDIT_RATIO'] = data['AMT_GOODS_PRICE'] / data['AMT_CREDIT']
data['EMPLOYED_TO_AGE_RATIO'] = data['DAYS_EMPLOYED'] / data['DAYS_BIRTH']

# --- EXT_SOURCE aggregates (giữ nguyên) ---
data['EXT_SOURCE_MEAN'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
data['EXT_SOURCE_PROD'] = data['EXT_SOURCE_1'] * data['EXT_SOURCE_2'] * data['EXT_SOURCE_3']
data['EXT_SOURCE_MIN'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
data['EXT_SOURCE_MAX'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)

# --- Social & Contact (giữ nguyên) ---
data['SOCIAL_DEF_TOTAL'] = data['DEF_30_CNT_SOCIAL_CIRCLE'] + data['DEF_60_CNT_SOCIAL_CIRCLE']

def age_group(age):
    if age < 27: return 0
    elif age < 35: return 1
    elif age < 45: return 2
    elif age < 55: return 3
    elif age < 65: return 4
    else: return 5

data['AGE_GROUP'] = data['AGE_YEARS'].apply(age_group)

contact_cols = ['FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
data['CONTACT_COUNT'] = data[contact_cols].sum(axis=1)

print(f'Baseline engineered: done')

In [ ]:
# ============================================================
# FEATURE ENGINEERING — MỚI (optimize)
# ============================================================

# --- 1. EXT_SOURCE interactions (TOP PRIORITY — chiếm >50% importance) ---
# Pairwise differences → captures relative strength between sources
data['EXT_SOURCE_1_2_DIFF'] = data['EXT_SOURCE_1'] - data['EXT_SOURCE_2']
data['EXT_SOURCE_2_3_DIFF'] = data['EXT_SOURCE_2'] - data['EXT_SOURCE_3']
data['EXT_SOURCE_1_3_DIFF'] = data['EXT_SOURCE_1'] - data['EXT_SOURCE_3']

# Pairwise products → interaction effects
data['EXT_SOURCE_1x2'] = data['EXT_SOURCE_1'] * data['EXT_SOURCE_2']
data['EXT_SOURCE_2x3'] = data['EXT_SOURCE_2'] * data['EXT_SOURCE_3']
data['EXT_SOURCE_1x3'] = data['EXT_SOURCE_1'] * data['EXT_SOURCE_3']

# Variance across sources → consistency of external scores
data['EXT_SOURCE_STD'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)

# Weighted mean (EXT_SOURCE_2 has highest importance)
data['EXT_SOURCE_WEIGHTED'] = (
    data['EXT_SOURCE_1'] * 0.2 + 
    data['EXT_SOURCE_2'] * 0.5 + 
    data['EXT_SOURCE_3'] * 0.3
)

# NaN count in EXT_SOURCE → how many sources are missing
data['EXT_SOURCE_NAN_COUNT'] = data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isnull().sum(axis=1)

# --- 2. Financial ratios (mở rộng) ---
# Annuity to goods price
data['ANNUITY_GOODS_RATIO'] = data['AMT_ANNUITY'] / data['AMT_GOODS_PRICE'].replace(0, np.nan)

# Credit vs goods difference (vay nhiều hơn giá trị → loan markup)
data['CREDIT_GOODS_DIFF'] = data['AMT_CREDIT'] - data['AMT_GOODS_PRICE']

# Monthly income (approximation)
data['INCOME_MONTHLY'] = data['AMT_INCOME_TOTAL'] / 12

# Months of income needed to repay entire credit
data['CREDIT_TO_MONTHLY_INCOME'] = data['AMT_CREDIT'] / (data['AMT_INCOME_TOTAL'] / 12)

# Annuity burden: annuity as % of monthly income
data['ANNUITY_BURDEN'] = data['AMT_ANNUITY'] / (data['AMT_INCOME_TOTAL'] / 12)

# --- 3. Age × Financial interactions ---
data['AGE_CREDIT_RATIO'] = data['AGE_YEARS'] / data['CREDIT_INCOME_RATIO'].replace(0, np.nan)
data['AGE_ANNUITY'] = data['AGE_YEARS'] * data['AMT_ANNUITY']

# --- 4. Employment stability ---
data['EMP_YEARS_BINNED'] = pd.cut(
    data['EMPLOYMENT_YEARS'], 
    bins=[-1, 0, 1, 3, 5, 10, 20, 100],
    labels=[0, 1, 2, 3, 4, 5, 6]
).astype(float)

# --- 5. Credit bureau inquiries ---
data['CREDIT_BUREAU_TOTAL'] = (
    data['AMT_REQ_CREDIT_BUREAU_HOUR'].fillna(0) +
    data['AMT_REQ_CREDIT_BUREAU_DAY'].fillna(0) +
    data['AMT_REQ_CREDIT_BUREAU_WEEK'].fillna(0) +
    data['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0) +
    data['AMT_REQ_CREDIT_BUREAU_QRT'].fillna(0) +
    data['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0)
)
# Recent inquiries are stronger signal
data['CREDIT_BUREAU_RECENT'] = (
    data['AMT_REQ_CREDIT_BUREAU_HOUR'].fillna(0) * 10 +
    data['AMT_REQ_CREDIT_BUREAU_DAY'].fillna(0) * 5 +
    data['AMT_REQ_CREDIT_BUREAU_WEEK'].fillna(0) * 3 +
    data['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0)
)

# --- 6. Region mismatch score ---
region_cols = ['REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION',
               'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY',
               'LIVE_CITY_NOT_WORK_CITY']
data['REGION_MISMATCH_SCORE'] = data[region_cols].sum(axis=1)

# --- 7. Document count ---
doc_cols = ['FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8']
data['DOCUMENT_COUNT'] = data[doc_cols].sum(axis=1)

# --- 8. Social observation ratio ---
data['SOCIAL_DEF_RATIO_30'] = (
    data['DEF_30_CNT_SOCIAL_CIRCLE'] / data['OBS_30_CNT_SOCIAL_CIRCLE'].replace(0, np.nan)
)
data['SOCIAL_DEF_RATIO_60'] = (
    data['DEF_60_CNT_SOCIAL_CIRCLE'] / data['OBS_60_CNT_SOCIAL_CIRCLE'].replace(0, np.nan)
)

# --- 9. EXT_SOURCE × Financial interactions ---
data['EXT2_CREDIT_RATIO'] = data['EXT_SOURCE_2'] * data['CREDIT_INCOME_RATIO']
data['EXT_MEAN_ANNUITY'] = data['EXT_SOURCE_MEAN'] * data['ANNUITY_INCOME_RATIO']

# --- 10. Night application flag ---
data['APPLIED_NIGHT'] = ((data['HOUR_APPR_PROCESS_START'] >= 22) | 
                          (data['HOUR_APPR_PROCESS_START'] <= 5)).astype(int)

print(f'Dataset shape after ALL engineering: {data.shape}')

# Count new features
new_engineered = [
    'EXT_SOURCE_1_2_DIFF', 'EXT_SOURCE_2_3_DIFF', 'EXT_SOURCE_1_3_DIFF',
    'EXT_SOURCE_1x2', 'EXT_SOURCE_2x3', 'EXT_SOURCE_1x3',
    'EXT_SOURCE_STD', 'EXT_SOURCE_WEIGHTED', 'EXT_SOURCE_NAN_COUNT',
    'ANNUITY_GOODS_RATIO', 'CREDIT_GOODS_DIFF', 'INCOME_MONTHLY',
    'CREDIT_TO_MONTHLY_INCOME', 'ANNUITY_BURDEN',
    'AGE_CREDIT_RATIO', 'AGE_ANNUITY',
    'EMP_YEARS_BINNED',
    'CREDIT_BUREAU_TOTAL', 'CREDIT_BUREAU_RECENT',
    'REGION_MISMATCH_SCORE', 'DOCUMENT_COUNT',
    'SOCIAL_DEF_RATIO_30', 'SOCIAL_DEF_RATIO_60',
    'EXT2_CREDIT_RATIO', 'EXT_MEAN_ANNUITY',
    'APPLIED_NIGHT',
]
print(f'\n+{len(new_engineered)} new engineered features')

## 5. Encode Categoricals

In [ ]:
# ============================================================
# ENCODE CATEGORICALS
# ============================================================
cat_cols = data.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    data[col] = data[col].fillna('MISSING')
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le
    print(f'  {col}: {len(le.classes_)} categories')

print(f'\nAll categoricals encoded.')

## 6. Final Feature Set

In [ ]:
# ============================================================
# FINAL FEATURE SET
# ============================================================
DROP_COLS = ['SK_ID_CURR', 'TARGET', 
             'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 
             'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE']

FINAL_FEATURES = [col for col in data.columns if col not in DROP_COLS]

print(f'Final feature count: {len(FINAL_FEATURES)}')
print(f'  Baseline raw:     {len(BASELINE_FEATURES)}')
print(f'  New raw:          {len(NEW_RAW_FEATURES)}')
print(f'  Engineered:       {len(FINAL_FEATURES) - len(BASELINE_FEATURES) - len(NEW_RAW_FEATURES) + 5}')  # +5 for dropped DAYS
print(f'\n--- Feature List ---')
for i, f in enumerate(FINAL_FEATURES, 1):
    marker = '  ' if f in BASELINE_FEATURES else ' ★' 
    print(f'  {i:2d}. {f}{marker}')
print(f'\n(★ = new/engineered feature)')

## 7. Prepare Data

In [ ]:
X = data[FINAL_FEATURES].copy()
y = data['TARGET'].copy()

print(f'X shape: {X.shape}')
print(f'y distribution: {y.value_counts().to_dict()}')
print(f'Positive rate: {y.mean():.4f}')

## 8. Hyperparameter Tuning — Optuna

Dùng Bayesian optimization để tìm hyperparameters tốt nhất cho LightGBM.
Chạy 50 trials, mỗi trial train 3-fold CV.

In [ ]:
# Cài optuna nếu chưa có
try:
    import optuna
    print(f'Optuna version: {optuna.__version__}')
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'optuna', '-q'])
    import optuna
    print(f'Optuna installed: {optuna.__version__}')

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ============================================================
# OPTUNA HYPERPARAMETER SEARCH
# ============================================================

def objective(trial):
    params = {
        'n_estimators': 2000,
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 80),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.01, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
        'is_unbalance': True,
    }
    
    # Quick 3-fold CV for speed
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    fold_aucs = []
    
    for fold_n, (train_idx, val_idx) in enumerate(folds.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric='auc',
            callbacks=[
                lgb.early_stopping(50, first_metric_only=True, verbose=False),
                lgb.log_evaluation(0),
            ]
        )
        val_preds = model.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, val_preds))
    
    return np.mean(fold_aucs)

# Run optimization
study = optuna.create_study(direction='maximize', study_name='lgbm_credit_scoring')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n{"="*60}')
print(f'Best AUC (3-fold): {study.best_value:.6f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print(f'{"="*60}')

In [ ]:
# Optuna visualization
try:
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except:
    # Fallback: print importance
    print('Parameter importances (from Optuna):')
    importances = optuna.importance.get_param_importances(study)
    for param, imp in importances.items():
        print(f'  {param}: {imp:.4f}')

## 9. Train Final Model — 5-Fold CV với Best Params

In [ ]:
# ============================================================
# FINAL TRAINING: 5-Fold CV with Optuna best params
# ============================================================
N_FOLDS = 5
folds = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Use best params from Optuna
best_params = study.best_params.copy()
best_params.update({
    'n_estimators': 2000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'is_unbalance': True,
})

oof_preds = np.zeros(len(X))
feature_importance_df = pd.DataFrame()
fold_aucs = []
best_model = None
best_auc = 0
models = []  # lưu tất cả models cho ensemble nếu cần

for fold_n, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = LGBMClassifier(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(100, first_metric_only=True, verbose=False),
            lgb.log_evaluation(0),
        ]
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f'Fold {fold_n + 1}: AUC = {fold_auc:.6f} | Best iter: {model.best_iteration_}')
    
    fold_imp = pd.DataFrame({
        'feature': FINAL_FEATURES,
        'importance': model.feature_importances_,
        'fold': fold_n + 1
    })
    feature_importance_df = pd.concat([feature_importance_df, fold_imp])
    models.append(model)
    
    if fold_auc > best_auc:
        best_auc = fold_auc
        best_model = model

overall_auc = roc_auc_score(y, oof_preds)
print(f'\n{"="*60}')
print(f'Overall OOF AUC:   {overall_auc:.6f}')
print(f'Mean Fold AUC:     {np.mean(fold_aucs):.6f} ± {np.std(fold_aucs):.6f}')
print(f'Baseline was:      0.7671')
print(f'Improvement:       +{(overall_auc - 0.7671):.4f} ({(overall_auc - 0.7671)/0.7671*100:.2f}%)')
print(f'{"="*60}')

## 10. Feature Importance — So sánh Old vs New

In [ ]:
# ============================================================
# FEATURE IMPORTANCE (top 30)
# ============================================================
mean_imp = (
    feature_importance_df
    .groupby('feature')['importance']
    .mean()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 14))
top30 = mean_imp.tail(30)

# Color: blue = baseline, orange = new
baseline_set = set(BASELINE_FEATURES)
colors = ['steelblue' if f in baseline_set else '#ff6b35' for f in top30.index]
top30.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 30 Feature Importance (🔵 Baseline  🟠 New)', fontsize=14)
ax.set_xlabel('Mean Importance')
plt.tight_layout()
plt.show()

# Print top 20
print('\nTop 20 features:')
for i, (feat, imp) in enumerate(mean_imp.tail(20).iloc[::-1].items(), 1):
    marker = '' if feat in baseline_set else ' ★NEW'
    print(f'  {i:2d}. {feat}: {imp:.0f}{marker}')

## 11. Feature Selection — Loại bỏ features yếu

Loại features có importance < threshold để giảm noise, có thể cải thiện AUC thêm.

In [ ]:
# ============================================================
# FEATURE SELECTION: Remove low-importance features
# ============================================================

# Loại features có mean importance < 1% of max importance
threshold = mean_imp.max() * 0.01
weak_features = mean_imp[mean_imp < threshold].index.tolist()

print(f'Threshold: {threshold:.1f}')
print(f'Weak features ({len(weak_features)}):')
for f in weak_features:
    print(f'  - {f}: {mean_imp[f]:.0f}')

# Train without weak features and compare
SELECTED_FEATURES = [f for f in FINAL_FEATURES if f not in weak_features]
print(f'\nFeatures: {len(FINAL_FEATURES)} → {len(SELECTED_FEATURES)} (removed {len(weak_features)})')

# Quick 3-fold test
X_selected = data[SELECTED_FEATURES].copy()
folds_quick = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
quick_aucs = []

for train_idx, val_idx in folds_quick.split(X_selected, y):
    m = LGBMClassifier(**best_params)
    m.fit(
        X_selected.iloc[train_idx], y.iloc[train_idx],
        eval_set=[(X_selected.iloc[val_idx], y.iloc[val_idx])],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
    )
    preds = m.predict_proba(X_selected.iloc[val_idx])[:, 1]
    quick_aucs.append(roc_auc_score(y.iloc[val_idx], preds))

print(f'\nWith ALL features:      AUC = {np.mean(fold_aucs):.6f}')
print(f'With SELECTED features: AUC = {np.mean(quick_aucs):.6f}')
print(f'Difference:             {np.mean(quick_aucs) - np.mean(fold_aucs):+.6f}')

# Decide which to use
if np.mean(quick_aucs) >= np.mean(fold_aucs) - 0.0005:
    USE_FEATURES = SELECTED_FEATURES
    print(f'\n✅ Using SELECTED features ({len(SELECTED_FEATURES)}) — simpler model, similar or better AUC')
else:
    USE_FEATURES = FINAL_FEATURES
    print(f'\n✅ Keeping ALL features ({len(FINAL_FEATURES)}) — feature selection hurt AUC')

## 12. FINAL TRAINING — với features đã chọn

Train 5-fold CV lần cuối với feature set tối ưu.

In [ ]:
# ============================================================
# FINAL TRAINING with selected features
# ============================================================
X_final = data[USE_FEATURES].copy()

N_FOLDS = 5
folds_final = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds_final = np.zeros(len(X_final))
final_fold_aucs = []
final_best_model = None
final_best_auc = 0
final_feature_importance = pd.DataFrame()

for fold_n, (train_idx, val_idx) in enumerate(folds_final.split(X_final, y)):
    X_train, X_val = X_final.iloc[train_idx], X_final.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = LGBMClassifier(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        eval_metric='auc',
        callbacks=[
            lgb.early_stopping(100, first_metric_only=True, verbose=False),
            lgb.log_evaluation(0),
        ]
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    oof_preds_final[val_idx] = val_preds
    
    fold_auc = roc_auc_score(y_val, val_preds)
    final_fold_aucs.append(fold_auc)
    print(f'Fold {fold_n + 1}: AUC = {fold_auc:.6f} | Best iter: {model.best_iteration_}')
    
    fold_imp = pd.DataFrame({
        'feature': USE_FEATURES,
        'importance': model.feature_importances_,
        'fold': fold_n + 1
    })
    final_feature_importance = pd.concat([final_feature_importance, fold_imp])
    
    if fold_auc > final_best_auc:
        final_best_auc = fold_auc
        final_best_model = model

final_auc = roc_auc_score(y, oof_preds_final)
print(f'\n{"="*60}')
print(f'🏆 FINAL OOF AUC:  {final_auc:.6f}')
print(f'   Mean Fold AUC:  {np.mean(final_fold_aucs):.6f} ± {np.std(final_fold_aucs):.6f}')
print(f'   Features used:  {len(USE_FEATURES)}')
print(f'\n   vs Old Baseline: 0.7671')
print(f'   Improvement:     +{(final_auc - 0.7671):.4f}')
print(f'{"="*60}')

## 13. SHAP Explainability

In [ ]:
try:
    import shap
    print(f'SHAP version: {shap.__version__}')
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'shap', '-q'])
    import shap
    print(f'SHAP installed: {shap.__version__}')

# SHAP on sample
sample_size = min(2000, len(X_final))
X_sample = X_final.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(final_best_model)
shap_values = explainer.shap_values(X_sample)

if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f'SHAP values computed for {sample_size} samples')

In [ ]:
plt.figure(figsize=(12, 12))
shap.summary_plot(shap_vals, X_sample, max_display=25, show=False)
plt.title('SHAP Summary — Optimized Model', fontsize=14)
plt.tight_layout()
plt.show()

## 14. Isotonic Calibration

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(oof_preds_final, y)
calibrated_preds = iso_reg.predict(oof_preds_final)

# Calibration comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

frac_before, mean_before = calibration_curve(y, oof_preds_final, n_bins=15)
axes[0].plot(mean_before, frac_before, 's-', color='#e74c3c', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Before Calibration')
axes[0].set_xlabel('Mean Predicted Prob')
axes[0].set_ylabel('Fraction of Positives')

frac_after, mean_after = calibration_curve(y, calibrated_preds, n_bins=15)
axes[1].plot(mean_after, frac_after, 's-', color='#2ecc71', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('After Isotonic Calibration')
axes[1].set_xlabel('Mean Predicted Prob')
axes[1].set_ylabel('Fraction of Positives')

plt.tight_layout()
plt.show()

print(f'AUC preserved: {roc_auc_score(y, calibrated_preds):.6f}')

## 15. Save Model Artifacts

In [ ]:
# ============================================================
# SAVE ARTIFACTS — Ghi đè vào model_artifacts/
# ============================================================
SAVE_DIR = Path('../model_artifacts')
SAVE_DIR.mkdir(exist_ok=True)

# 1. Model
joblib.dump(final_best_model, SAVE_DIR / 'lgbm_credit_scoring.pkl')
print(f'✅ Model saved ({(SAVE_DIR / "lgbm_credit_scoring.pkl").stat().st_size / 1024:.0f} KB)')

# 2. Feature names (đúng thứ tự!)
with open(SAVE_DIR / 'feature_names.json', 'w') as f:
    json.dump(USE_FEATURES, f, indent=2)
print(f'✅ Feature names saved ({len(USE_FEATURES)} features)')

# 3. Label encoders
joblib.dump(label_encoders, SAVE_DIR / 'label_encoders.pkl')
print(f'✅ Label encoders saved ({len(label_encoders)} encoders)')

# 4. SHAP explainer
joblib.dump(explainer, SAVE_DIR / 'shap_explainer.pkl')
print(f'✅ SHAP explainer saved')

# 5. Isotonic calibrator
joblib.dump(iso_reg, SAVE_DIR / 'isotonic_calibrator.pkl')
print(f'✅ Isotonic calibrator saved')

# 6. Feature descriptions (update with new features)
FEATURE_DESCRIPTIONS = {
    # User-facing fields (same as baseline)
    'CODE_GENDER': {'label': 'Giới tính', 'type': 'select', 'options': ['F', 'M'], 'required': True},
    'AGE_YEARS': {'label': 'Tuổi', 'type': 'number', 'min': 18, 'max': 80, 'required': True},
    'CNT_CHILDREN': {'label': 'Số con', 'type': 'number', 'min': 0, 'max': 20, 'required': True},
    'CNT_FAM_MEMBERS': {'label': 'Số thành viên gia đình', 'type': 'number', 'min': 1, 'max': 30, 'required': True},
    'NAME_EDUCATION_TYPE': {'label': 'Trình độ học vấn', 'type': 'select', 
                            'options': ['Lower secondary', 'Secondary / secondary special', 
                                       'Incomplete higher', 'Higher education', 'Academic degree'],
                            'required': True},
    'NAME_FAMILY_STATUS': {'label': 'Tình trạng hôn nhân', 'type': 'select',
                           'options': ['Single / not married', 'Married', 'Civil marriage', 
                                      'Separated', 'Widow'],
                           'required': True},
    'AMT_INCOME_TOTAL': {'label': 'Thu nhập hàng năm', 'type': 'number', 'required': True},
    'AMT_CREDIT': {'label': 'Số tiền muốn vay', 'type': 'number', 'required': True},
    'AMT_ANNUITY': {'label': 'Số tiền trả hàng tháng', 'type': 'number', 'required': True},
    'AMT_GOODS_PRICE': {'label': 'Giá trị hàng hóa/mục đích vay', 'type': 'number', 'required': True},
    'NAME_CONTRACT_TYPE': {'label': 'Loại hợp đồng', 'type': 'select', 
                           'options': ['Cash loans', 'Revolving loans'], 'required': True},
    'EMPLOYMENT_YEARS': {'label': 'Số năm đi làm', 'type': 'number', 'min': 0, 'max': 60, 'required': True},
    'NAME_INCOME_TYPE': {'label': 'Loại thu nhập', 'type': 'select',
                         'options': ['Working', 'Commercial associate', 'Pensioner', 
                                    'State servant', 'Businessman', 'Student', 'Unemployed',
                                    'Maternity leave'],
                         'required': True},
    'OCCUPATION_TYPE': {'label': 'Nghề nghiệp', 'type': 'select',
                        'options': ['Laborers', 'Sales staff', 'Core staff', 'Managers',
                                   'Drivers', 'High skill tech staff', 'Accountants',
                                   'Medicine staff', 'Cooking staff', 'Security staff',
                                   'Cleaning staff', 'Private service staff', 'Low-skill Laborers',
                                   'Waiters/barmen staff', 'Secretaries', 'Realty agents', 'IT staff',
                                   'HR staff'],
                        'required': False},
    'FLAG_OWN_CAR': {'label': 'Sở hữu xe hơi', 'type': 'select', 'options': ['Y', 'N'], 'required': True},
    'FLAG_OWN_REALTY': {'label': 'Sở hữu bất động sản', 'type': 'select', 'options': ['Y', 'N'], 'required': True},
    'OWN_CAR_AGE': {'label': 'Tuổi xe (năm)', 'type': 'number', 'min': 0, 'max': 80, 'required': False},
    'NAME_HOUSING_TYPE': {'label': 'Loại nhà ở', 'type': 'select',
                          'options': ['House / apartment', 'With parents', 'Municipal apartment',
                                     'Rented apartment', 'Office apartment', 'Co-op apartment'],
                          'required': True},
    'EXT_SOURCE_1': {'label': 'Điểm tín dụng thay thế 1', 'type': 'number', 'min': 0, 'max': 1, 'required': False},
    'EXT_SOURCE_2': {'label': 'Điểm tín dụng thay thế 2', 'type': 'number', 'min': 0, 'max': 1, 'required': False},
    'EXT_SOURCE_3': {'label': 'Điểm tín dụng thay thế 3', 'type': 'number', 'min': 0, 'max': 1, 'required': False},
}

with open(SAVE_DIR / 'feature_descriptions.json', 'w', encoding='utf-8') as f:
    json.dump(FEATURE_DESCRIPTIONS, f, indent=2, ensure_ascii=False)
print(f'✅ Feature descriptions saved')

# 7. Save Optuna best params for reference
with open(SAVE_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print(f'✅ Best params saved')

print(f'\n📁 All artifacts saved to: {SAVE_DIR.resolve()}')

## 16. Model Summary — Comparison

In [ ]:
import os

model_size = os.path.getsize(SAVE_DIR / 'lgbm_credit_scoring.pkl') / 1024

print('╔════════════════════════════════════════════════════════════╗')
print('║     CREDIT SCORING MODEL — OPTIMIZATION RESULTS          ║')
print('╠════════════════════════════════════════════════════════════╣')
print(f'║                                                            ║')
print(f'║  {"Metric":<25} {"Old Baseline":>14} {"Optimized":>14}  ║')
print(f'║  {"─"*25} {"─"*14} {"─"*14}  ║')
print(f'║  {"OOF AUC":<25} {"0.7671":>14} {f"{final_auc:.4f}":>14}  ║')
print(f'║  {"Features":<25} {"48":>14} {f"{len(USE_FEATURES)}":>14}  ║')
print(f'║  {"Data sources":<25} {"1 table":>14} {"1 table":>14}  ║')
print(f'║  {"CV Folds":<25} {"5":>14} {"5":>14}  ║')
print(f'║  {"HP Tuning":<25} {"Manual":>14} {"Optuna 50t":>14}  ║')
print(f'║  {"Model size":<25} {"~1.8 MB":>14} {f"{model_size:.0f} KB":>14}  ║')
print(f'║                                                            ║')
print(f'║  Improvement: +{(final_auc - 0.7671):.4f} AUC ({(final_auc - 0.7671)/0.7671*100:.2f}%)              ║')
print(f'║                                                            ║')
print('╚════════════════════════════════════════════════════════════╝')

print(f'\n📌 Optimizations applied:')
print(f'  1. ✅ Added {len(NEW_RAW_FEATURES)} new raw features from application_train')
print(f'  2. ✅ Added ~26 new engineered features (EXT interactions, bureau, etc.)')
print(f'  3. ✅ Optuna hyperparameter tuning (50 trials)')
print(f'  4. ✅ Feature selection (removed weak features)')
print(f'  5. ✅ Isotonic probability calibration')

## 17. ⚠️ Lưu ý khi update Webapp

Sau khi chạy notebook này, model artifacts được ghi đè. Webapp (`engine.py`) cần update:

1. **`engine.py` → `_build_feature_row()`**: Thêm logic cho các features mới
2. **`app.py` → form input**: Có thể KHÔNG cần thêm field mới (hầu hết features mới là engineered hoặc default NaN)
3. **`config.py` → `FEATURE_LABELS_VI`**: Thêm labels cho features mới

Xem hướng dẫn chi tiết trong bảng dưới:

In [ ]:
# ============================================================
# PHÂN LOẠI FEATURES: nào cần input, nào tự tính
# ============================================================

# Features user nhập trên form (same as before)
USER_INPUT_FEATURES = {
    'CODE_GENDER', 'AGE_YEARS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_CONTRACT_TYPE', 'EMPLOYMENT_YEARS', 'NAME_INCOME_TYPE',
    'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'OWN_CAR_AGE',
    'NAME_HOUSING_TYPE', 
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE',
    'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL',
}

# Features tính từ input (engine.py tự tính)
ENGINEERED_FEATURES = {
    'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_TERM_MONTHS',
    'PAYMENT_RATE', 'INCOME_PER_PERSON', 'GOODS_CREDIT_RATIO',
    'EMPLOYED_TO_AGE_RATIO', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_PROD',
    'EXT_SOURCE_MIN', 'EXT_SOURCE_MAX', 'SOCIAL_DEF_TOTAL',
    'AGE_GROUP', 'CONTACT_COUNT',
    # NEW engineered
    'EXT_SOURCE_1_2_DIFF', 'EXT_SOURCE_2_3_DIFF', 'EXT_SOURCE_1_3_DIFF',
    'EXT_SOURCE_1x2', 'EXT_SOURCE_2x3', 'EXT_SOURCE_1x3',
    'EXT_SOURCE_STD', 'EXT_SOURCE_WEIGHTED', 'EXT_SOURCE_NAN_COUNT',
    'ANNUITY_GOODS_RATIO', 'CREDIT_GOODS_DIFF', 'INCOME_MONTHLY',
    'CREDIT_TO_MONTHLY_INCOME', 'ANNUITY_BURDEN',
    'AGE_CREDIT_RATIO', 'AGE_ANNUITY', 'EMP_YEARS_BINNED',
    'CREDIT_BUREAU_TOTAL', 'CREDIT_BUREAU_RECENT',
    'REGION_MISMATCH_SCORE', 'DOCUMENT_COUNT',
    'SOCIAL_DEF_RATIO_30', 'SOCIAL_DEF_RATIO_60',
    'EXT2_CREDIT_RATIO', 'EXT_MEAN_ANNUITY', 'APPLIED_NIGHT',
}

# Features mặc định NaN (model xử lý được, KHÔNG cần form input)
DEFAULT_NAN_FEATURES = set(USE_FEATURES) - USER_INPUT_FEATURES - ENGINEERED_FEATURES

print(f'Feature breakdown:')
print(f'  User input (form):  {len(USER_INPUT_FEATURES & set(USE_FEATURES))}')
print(f'  Engineered (auto):  {len(ENGINEERED_FEATURES & set(USE_FEATURES))}')
print(f'  Default NaN:        {len(DEFAULT_NAN_FEATURES)}')
print(f'  TOTAL:              {len(USE_FEATURES)}')

if DEFAULT_NAN_FEATURES:
    print(f'\nDefault NaN features (model handles missing, no form needed):')
    for f in sorted(DEFAULT_NAN_FEATURES):
        print(f'  - {f}')